# SurfaceTarget-CRC — phase 04 — patch 4 1

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1
#
# DEPMAP RELEASE AND INPUT DATA AUDIT
#
# Objectives:
# 1. Validate Phase 3 candidate evidence
# 2. Establish DepMap 26Q3 as the intended release
# 3. Check official DepMap release metadata
# 4. Prepare directories for CRISPR and model data
# 5. Detect and inspect any existing source files
# 6. Preserve a reproducible candidate reference
# 7. Save the Phase 4 research plan and provenance
#
# IMPORTANT:
# - No CRISPR dependency analysis yet
# - No patient-level TCGA/CPTAC/DepMap merging
# - No automatic use of older DepMap releases
# - No bypassing DepMap browser verification
# - No candidate exclusions
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests

import hashlib
import json
import io
import re


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"

DEPMAP_RAW = (
    ROOT / "data/raw/depmap/26Q3"
)

for directory in [
    META,
    RESULTS,
    DEPMAP_RAW
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

CANDIDATE_PATH = (
    RESULTS /
    "surfacetarget_normal_tissue_integration.csv"
)

RELEASE = "26Q3"

REQUIRED_FILES = [
    "Model.csv",
    "CRISPRGeneEffect.csv"
]

CATALOGUE_URL = (
    "https://depmap.org/portal/"
    "api/no-captcha/download/files"
)

PORTAL_URL = (
    "https://depmap.org/portal/data_page/"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.1")
print("DEPMAP RELEASE AND INPUT AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load current research candidates
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING EXISTING CANDIDATE EVIDENCE")
print("-" * 65)

if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(
        "Phase 3 integrated candidate table missing."
    )

candidates = pd.read_csv(
    CANDIDATE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)

required_columns = {
    "gene_id",
    "gene_name",
    "ensembl_stable_id",
    "protein_enriched",
    "uniprot_accession",
    "uniprot_topology_category"
}

missing_columns = (
    required_columns - set(candidates.columns)
)

if missing_columns:
    raise RuntimeError(
        f"Missing candidate columns: {missing_columns}"
    )

if len(candidates) != 948:
    raise RuntimeError(
        "Expected 948 candidate genes."
    )

if candidates["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate transcriptomic gene IDs."
    )

if candidates[
    "ensembl_stable_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate stable Ensembl IDs."
    )

protein_supported = (
    candidates["protein_enriched"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

if int(protein_supported.sum()) != 20:
    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )

print("\nCandidate table validated.")

print("Transcriptomic candidates:", len(candidates))

print(
    "Cross-omics-supported candidates:",
    int(protein_supported.sum())
)


# ------------------------------------------------------------
# 3. Freeze Phase 4 gene reference
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING DEPMAP GENE REFERENCE")
print("-" * 65)

reference = candidates[
    [
        "gene_id",
        "ensembl_stable_id",
        "gene_name",
        "protein_enriched",
        "uniprot_accession",
        "uniprot_topology_category"
    ]
].copy()

reference["protein_enriched"] = (
    protein_supported.to_numpy(dtype=bool)
)

reference["gene_name"] = (
    reference["gene_name"]
    .astype("string")
    .str.strip()
)

missing_symbols = int(
    reference["gene_name"].isna().sum()
)

duplicate_symbols = int(
    reference["gene_name"]
    .dropna()
    .duplicated()
    .sum()
)

print("\nGenes in reference:", len(reference))

print("Missing gene symbols:", missing_symbols)

print(
    "Repeated gene symbols:",
    duplicate_symbols
)

print(
    "\nGene symbols are retained for inspection."
)

print(
    "DepMap gene identifiers will be validated "
    "before any CRISPR integration."
)

REFERENCE_PATH = (
    META / "depmap_phase4_candidate_reference.csv"
)

reference.to_csv(
    REFERENCE_PATH,
    index=False
)


# ------------------------------------------------------------
# 4. Retrieve official release catalogue
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CHECKING OFFICIAL DEPMAP RELEASE METADATA")
print("-" * 65)

catalogue = None
catalogue_status = "Unavailable"

try:

    response = requests.get(
        CATALOGUE_URL,
        headers={
            "User-Agent":
                "SurfaceTarget-CRC-Academic-Research/1.0",
            "Accept": "text/csv"
        },
        timeout=(15, 45)
    )

    response.raise_for_status()

    beginning = response.text[:1000].lower()

    if (
        "<html" in beginning
        or "<!doctype html" in beginning
    ):
        raise ValueError(
            "DepMap returned browser verification "
            "instead of release metadata."
        )

    parsed = pd.read_csv(
        io.StringIO(response.text),
        dtype=str
    )

    required_catalogue_columns = {
        "release",
        "filename"
    }

    if not required_catalogue_columns.issubset(
        parsed.columns
    ):
        raise ValueError(
            "Unexpected DepMap catalogue schema."
        )

    catalogue = parsed

    catalogue_status = "Retrieved"

    print("\nOfficial catalogue retrieved.")

    print("Catalogue records:", len(catalogue))

except Exception as error:

    print("\nOfficial catalogue unavailable.")

    print("Reason:", str(error))

    print(
        "This does not indicate that "
        "the DepMap release is unavailable."
    )


# ------------------------------------------------------------
# 5. Inspect intended release
# ------------------------------------------------------------

release_files = pd.DataFrame()

if catalogue is not None:

    release_mask = (
        catalogue["release"]
        .astype(str)
        .str.contains(
            r"26Q3",
            case=False,
            regex=True,
            na=False
        )
    )

    release_rows = catalogue[
        release_mask
    ].copy()

    release_files = release_rows[
        release_rows["filename"].isin(
            REQUIRED_FILES
        )
    ].copy()

    print("\n26Q3 catalogue entries:")

    print(
        release_files.to_string(
            index=False
        )
    )

    release_files.to_csv(
        META / "depmap_26q3_source_catalogue.csv",
        index=False
    )

    for filename in REQUIRED_FILES:

        matches = release_files[
            release_files["filename"].eq(filename)
        ]

        print(
            f"{filename}: "
            f"{len(matches)} catalogue entry/entries"
        )


# ------------------------------------------------------------
# 6. Audit local DepMap files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEPMAP INPUT FILE AVAILABILITY")
print("-" * 65)

input_records = []

for filename in REQUIRED_FILES:

    path = DEPMAP_RAW / filename

    exists = path.exists()

    size_mib = (
        path.stat().st_size / (1024**2)
        if exists else np.nan
    )

    header_columns = []

    header_valid = False

    validation_note = "Not yet provided"

    if exists:

        try:

            preview = pd.read_csv(
                path,
                nrows=2,
                low_memory=False
            )

            header_columns = (
                preview.columns.astype(str).tolist()
            )

            header_valid = (
                "ModelID" in header_columns
            )

            if not header_valid:

                validation_note = (
                    "ModelID column not found; "
                    "requires review"
                )

            elif filename == "CRISPRGeneEffect.csv":

                if len(header_columns) < 1000:

                    header_valid = False

                    validation_note = (
                        "Unexpectedly few gene columns"
                    )

                else:

                    validation_note = (
                        "Preliminary header validation passed"
                    )

            else:

                validation_note = (
                    "Preliminary header validation passed"
                )

        except Exception as error:

            validation_note = str(error)

    input_records.append({

        "release": RELEASE,

        "filename": filename,

        "local_path": str(path),

        "available": exists,

        "size_mib": size_mib,

        "header_valid": header_valid,

        "header_preview":
            "; ".join(header_columns[:8]),

        "validation_note": validation_note

    })


input_audit = pd.DataFrame(
    input_records
)

print("\n")

print(
    input_audit[
        [
            "filename",
            "available",
            "size_mib",
            "header_valid",
            "validation_note"
        ]
    ].to_string(index=False)
)

input_audit.to_csv(
    META / "depmap_26q3_input_audit.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Record analysis strategy
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEPMAP ANALYSIS STRATEGY")
print("-" * 65)

analysis_strategy = {

    "primary_focus":
        "Colorectal cancer cell models",

    "model_annotation_source":
        "DepMap Public 26Q3 Model.csv",

    "dependency_source":
        "DepMap Public 26Q3 CRISPRGeneEffect.csv",

    "model_selection": (
        "Identify colorectal models using "
        "the release's actual OncoTree metadata; "
        "review colon and rectal subgroups."
    ),

    "msi_handling": (
        "Do not assign MSS status without "
        "verified model-specific MSI annotations."
    ),

    "dependency_interpretation": (
        "More negative Chronos gene-effect "
        "scores indicate greater knockout "
        "fitness effects."
    ),

    "primary_analyses": [
        "Colorectal model dependency distribution",
        "Fraction of models with strong negative effects",
        "Comparison with noncolorectal lineages",
        "Cancer-lineage selectivity assessment",
        "Candidate-level coverage audit"
    ],

    "important_limitation": (
        "CRISPR fitness dependency is "
        "not a necessary condition for "
        "antibody-mediated therapeutic targeting."
    )
}


# ------------------------------------------------------------
# 8. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "intended_depmap_release":
        "DepMap Public 26Q3",

    "official_download_page": PORTAL_URL,

    "official_catalogue_url": CATALOGUE_URL,

    "catalogue_status": catalogue_status,

    "candidate_count": int(
        len(reference)
    ),

    "cross_omics_supported_count": 20,

    "input_files_required": REQUIRED_FILES,

    "input_files_available": int(
        input_audit["available"].sum()
    ),

    "preliminary_validated_files": int(
        input_audit["header_valid"].sum()
    ),

    "analysis_strategy": analysis_strategy,

    "crispr_analysis_performed": False,

    "target_ranking_performed": False,

    "candidate_exclusions": 0
}


with open(
    META / "depmap_phase4_setup_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1 SUMMARY")
print("=" * 65)

print("\nIntended release: DepMap Public 26Q3")

print("Candidate genes:", len(reference))

print("Required DepMap files:", len(REQUIRED_FILES))

print(
    "Files currently available:",
    int(input_audit["available"].sum())
)

print(
    "Files passing preliminary checks:",
    int(input_audit["header_valid"].sum())
)

print("\nExpected input directory:")
print(DEPMAP_RAW)

print("\nResearch reference: SAVED")

print("Input audit: SAVED")

print("Analysis strategy: SAVED")

print("Provenance: SAVED")

print("\nNo dependency statistics calculated.")
print("No candidate genes excluded.")
print("No previous research files modified.")

if not input_audit["header_valid"].all():

    print("\nNEXT ACTION:")
    print(
        "Obtain official 26Q3 Model.csv and "
        "CRISPRGeneEffect.csv files."
    )

else:

    print("\nBoth required inputs are available.")
    print("Ready for full source verification.")

print("\nPATCH 4.1 COMPLETE")
print("=" * 65)